# Unlimited-OCR — Colab

Model: [Oguzbayrakdar/Unlimited-OCR](https://huggingface.co/Oguzbayrakdar/Unlimited-OCR) (kopyası alınan kaynak: [baidu/Unlimited-OCR](https://huggingface.co/baidu/Unlimited-OCR), MIT lisansı)

**Kullanım**
1. Menüden **Runtime → Change runtime type → T4 GPU** seçin.
2. **Runtime → Run all** ile tüm hücreleri çalıştırın.
3. Son hücrenin altında çıkan `https://….gradio.live` linkini açın, resim veya PDF yükleyin.

İlk çalıştırmada model (~6.7 GB) indirileceği için birkaç dakika sürer. Link, notebook açık kaldığı sürece çalışır.

In [ ]:
# GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Bağımlılıklar. transformers 4.57.1 modelin test edildiği sürüm;
# gradio 6.x, transformers 4.57.1 ile çakışan bir huggingface_hub istediği için 5.x kullanılıyor.
!pip install -q "transformers==4.57.1" "gradio>=5,<6" einops addict easydict pymupdf matplotlib accelerate

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "Oguzbayrakdar/Unlimited-OCR"

assert torch.cuda.is_available(), "GPU bulunamadı: Runtime → Change runtime type → T4 GPU"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
).eval().cuda()
print("Model hazır.")

In [ ]:
import os
import tempfile

import fitz  # pymupdf
import gradio as gr

MODES = {
    "gundam (hızlı)": dict(base_size=1024, image_size=640, crop_mode=True),
    "base (daha doğru)": dict(base_size=1024, image_size=1024, crop_mode=False),
}


def pdf_to_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    out_dir = tempfile.mkdtemp(prefix="pdf_")
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []
    for i, page in enumerate(doc):
        path = os.path.join(out_dir, f"page_{i + 1:04d}.png")
        page.get_pixmap(matrix=mat).save(path)
        paths.append(path)
    doc.close()
    return paths


def ocr_image(image_path, mode):
    out_dir = tempfile.mkdtemp(prefix="ocr_")
    model.infer(
        tokenizer,
        prompt="<image>document parsing.",
        image_file=image_path,
        output_path=out_dir,
        max_length=8192,
        no_repeat_ngram_size=35,
        ngram_window=128,
        save_results=True,
        **MODES[mode],
    )
    with open(os.path.join(out_dir, "result.md"), encoding="utf-8") as f:
        return f.read().strip()


def run(file_path, mode, progress=gr.Progress()):
    if not file_path:
        raise gr.Error("Önce bir resim veya PDF yükleyin.")
    pages = pdf_to_images(file_path) if file_path.lower().endswith(".pdf") else [file_path]

    parts = []
    for i, page in enumerate(progress.tqdm(pages, desc="Sayfa")):
        text = ocr_image(page, mode)
        parts.append(f"<!-- Sayfa {i + 1} -->\n\n{text}" if len(pages) > 1 else text)
        yield "\n\n---\n\n".join(parts), None

    result = "\n\n---\n\n".join(parts)
    out_path = os.path.join(tempfile.mkdtemp(), "sonuc.md")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(result)
    yield result, out_path


with gr.Blocks(title="Unlimited-OCR") as demo:
    gr.Markdown("# Unlimited-OCR\nResim veya PDF yükleyin; çıktı Markdown olarak gelir.")
    with gr.Row():
        with gr.Column(scale=1):
            inp = gr.File(label="Belge", file_types=[".pdf", ".png", ".jpg", ".jpeg", ".webp", ".bmp"], type="filepath")
            mode = gr.Radio(list(MODES), value="gundam (hızlı)", label="Mod")
            btn = gr.Button("Çalıştır", variant="primary")
            out_file = gr.File(label="Sonucu indir (.md)")
        with gr.Column(scale=2):
            out_md = gr.Markdown(label="Sonuç")
    btn.click(run, inputs=[inp, mode], outputs=[out_md, out_file])

demo.queue().launch(share=True, debug=True)